# US-04-04 — Hazelnut EXP-002 — Pooling corrigé

Export Kaggle original avec sorties conservées. Seule cette introduction a été ajoutée. La parité des embeddings est attestée sur un batch de deux normales ; la restauration CPU indépendante est consignée dans les sorties et le rapport. Ces contrôles ne mesurent pas les performances de détection.

Voir [la fiche US-04-04](../docs/epic-04/us-04-04.md). Le commit 9c93258 identifie le pipeline commun, pas le code de cet export. Aucun identifiant de version Kaggle n’est disponible ; les empreintes des exports sont enregistrées dans la provenance. Le JSON de configuration Git est descriptif : les cellules conservent leurs paramètres originaux.

Préparer une session T4 avec Internet et MVTec AD v2. Les métadonnées Quick Save indiquent aucun accélérateur/input et Internet désactivé, malgré les sorties GPU : configurer la session explicitement.


In [1]:

import os
import sys
import random
import subprocess
import importlib.metadata as metadata

import numpy as np
import torch
import torchvision

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=== HAZELNUT ENVIRONMENT ===")
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())

for package in ["anomalib", "timm", "lightning"]:
    try:
        print(package, metadata.version(package))
    except metadata.PackageNotFoundError:
        print(package, "NOT INSTALLED")

assert os.path.isdir("/kaggle")
assert torch.cuda.is_available()

print("GPU:", torch.cuda.get_device_name(0))
print("Seed:", SEED)
print("\nEnvironment preflight: OK")


=== HAZELNUT ENVIRONMENT ===
Python: 3.13.16
Torch: 2.11.0+cu130
Torchvision: 0.26.0+cu130
CUDA: 13.0
GPU available: True
anomalib NOT INSTALLED
timm 1.0.30
lightning NOT INSTALLED
GPU: Tesla T4
Seed: 42

Environment preflight: OK


In [2]:
import csv
import hashlib
import subprocess
from pathlib import Path

REPO_DIR = Path("/kaggle/working/industrial-defect-ai")

SOURCE_COMMIT = (
    "9c932580cbbe27c61d92905e344c8c7e86457381"
)

if not (REPO_DIR / ".git").is_dir():
    subprocess.run(
        [
            "git", "clone", "--quiet",
            "https://github.com/MRdrissi/industrial-defect-ai.git",
            str(REPO_DIR)
        ],
        check=True
    )

subprocess.run(
    [
        "git", "-C", str(REPO_DIR),
        "checkout", "--detach", SOURCE_COMMIT
    ],
    check=True
)

manifest = (
    REPO_DIR / "docs/data/us-02-03/hazelnut_manifest.csv"
)

with manifest.open(encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f))

train = [r for r in rows if r["split"] == "train"]
val = [r for r in rows if r["split"] == "val"]
test = [r for r in rows if r["split"] == "test"]

train_paths = {r["image_path"] for r in train}
val_paths = {r["image_path"] for r in val}
test_paths = {r["image_path"] for r in test}

assert len(train) == 313
assert len(val) == 78
assert len(test) == 110

assert len(train_paths) == len(train)
assert len(val_paths) == len(val)
assert len(test_paths) == len(test)

assert train_paths.isdisjoint(val_paths)
assert train_paths.isdisjoint(test_paths)
assert val_paths.isdisjoint(test_paths)

assert all(
    r["category"] == "hazelnut"
    and str(r["label"]).strip() == "0"
    for r in train
)

manifest_sha = hashlib.sha256(
    manifest.read_bytes()
).hexdigest()

print("=== HAZELNUT DATA PROTOCOL ===")
print("Train normal:", len(train))
print("Validation normal:", len(val))
print("Test official:", len(test))
print("Manifest SHA256:", manifest_sha)
print("Git commit:", SOURCE_COMMIT)

print("\nNo data leakage: OK")
print("Hazelnut manifest validation: OK")

=== HAZELNUT DATA PROTOCOL ===
Train normal: 313
Validation normal: 78
Test official: 110
Manifest SHA256: 8f0e0d5b5bb767cd2d2e6f8a0ab10f65a959a97a38bb14bc5d9aae1d30c81fb6
Git commit: 9c932580cbbe27c61d92905e344c8c7e86457381

No data leakage: OK
Hazelnut manifest validation: OK


HEAD is now at 9c93258 docs(EPIC-04): register completed US-04-02


In [3]:
import sys
import subprocess
import json
from pathlib import Path

plan_path = Path("/kaggle/working/anomalib_install_plan.json")

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--dry-run", "--report", str(plan_path),
        "anomalib==2.7.0"
    ],
    check=True
)

plan = json.loads(plan_path.read_text())

planned = {
    item["metadata"]["name"].lower().replace("_", "-")
    for item in plan["install"]
}

print("Packages to install:", sorted(planned))

protected = {"torch", "torchvision", "triton"}
assert not planned.intersection(protected), (
    "Installation would modify PyTorch dependencies"
)

print("\nDependency plan: OK")

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Would install FrEIA-0.2 anomalib-2.7.0 imagecodecs-2026.11.11 jsonargparse-4.46.0 lightning-2.6.6 rich-argparse-1.8.0 typeshed_client-2.15.0
Packages to install: ['anomalib', 'freia', 'imagecodecs', 'jsonargparse', 'lightning', 'rich-argparse', 'typeshed-client']

Dependency plan: OK


In [4]:

import subprocess
import sys

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "anomalib==2.7.0"
    ],
    check=True
)

print("Anomalib installation: OK")


  Using cached anomalib-2.7.0-py3-none-any.whl.metadata (35 kB)
  Using cached FrEIA-0.2.tar.gz (34 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached imagecodecs-2026.11.11-cp312-abi3-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached jsonargparse-4.46.0-py3-none-any.whl.metadata (12 kB)
  Using cached lightning-2.6.6-py3-none-any.whl.metadata (45 kB)
  Using cached rich_argparse-1.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached typeshed_client-2.15.0-py3-none-any.whl.metadata (11 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 30.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 808

In [5]:

import importlib.metadata as metadata
import torch
import torchvision

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Anomalib:", metadata.version("anomalib"))
print("Lightning:", metadata.version("lightning"))
print("CUDA:", torch.cuda.is_available())

assert torch.__version__ == "2.11.0+cu130"
assert torchvision.__version__ == "0.26.0+cu130"
assert metadata.version("anomalib") == "2.7.0"
assert torch.cuda.is_available()

print("\nHazelnut dependencies: OK")


Torch: 2.11.0+cu130
Torchvision: 0.26.0+cu130
Anomalib: 2.7.0
Lightning: 2.6.6
CUDA: True

Hazelnut dependencies: OK


In [6]:

import sys
import csv
import hashlib
import subprocess
from pathlib import Path

import torch

REPO_DIR = Path("/kaggle/working/industrial-defect-ai")
SOURCE_COMMIT = "9c932580cbbe27c61d92905e344c8c7e86457381"
DATASET_ROOT = Path("/kaggle/input/datasets/ipythonx/mvtec-ad")

# Reproduce the project code at a fixed commit
if not (REPO_DIR / ".git").exists():
    subprocess.run(
        [
            "git", "clone", "--quiet",
            "https://github.com/MRdrissi/industrial-defect-ai.git",
            str(REPO_DIR)
        ],
        check=True
    )

subprocess.run(
    [
        "git", "-C", str(REPO_DIR),
        "checkout", "--detach", SOURCE_COMMIT
    ],
    check=True
)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.data.mvtec_dataset import MVTecDataset
from src.data.transforms import MVTecTransform
from src.data.dataloaders import create_dataloader

MANIFEST = (
    REPO_DIR / "docs/data/us-02-03/hazelnut_manifest.csv"
)

assert MANIFEST.is_file()
assert DATASET_ROOT.is_dir()

# Verify logical splits
with MANIFEST.open(encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f))

train_rows = [r for r in rows if r["split"] == "train"]
val_rows = [r for r in rows if r["split"] == "val"]
test_rows = [r for r in rows if r["split"] == "test"]

train_paths = {r["image_path"] for r in train_rows}
val_paths = {r["image_path"] for r in val_rows}
test_paths = {r["image_path"] for r in test_rows}

assert len(train_rows) == len(train_paths) == 313
assert len(val_rows) == len(val_paths) == 78
assert len(test_rows) == len(test_paths) == 110

assert train_paths.isdisjoint(val_paths)
assert train_paths.isdisjoint(test_paths)
assert val_paths.isdisjoint(test_paths)

assert all(
    r["category"] == "hazelnut"
    and int(r["label"]) == 0
    for r in train_rows
)

# Reuse US-02-04 preprocessing
transform = MVTecTransform(
    size=(256, 256),
    profile="patchcore"
)

train_dataset = MVTecDataset(
    dataset_root=DATASET_ROOT,
    manifest_path=MANIFEST,
    category="hazelnut",
    split="train",
    transform=transform
)

train_loader = create_dataloader(
    train_dataset,
    batch_size=2,
    num_workers=0,
    seed=42
)

batch = next(iter(train_loader))

assert len(train_dataset) == 313
assert batch["image"].shape == (2, 3, 256, 256)
assert torch.all(batch["label"] == 0)
assert torch.isfinite(batch["image"]).all()
assert set(batch["image_path"]).issubset(train_paths)

print("=== HAZELNUT DATA VALIDATION ===")
print("Train normal:", len(train_rows))
print("Validation normal:", len(val_rows))
print("Test official:", len(test_rows))
print("Batch shape:", tuple(batch["image"].shape))
print("Paths:", batch["image_path"])
print("Manifest SHA256:",
      hashlib.sha256(MANIFEST.read_bytes()).hexdigest())

print("\nNo data leakage: OK")
print("Hazelnut pipeline: OK")


HEAD is now at 9c93258 docs(EPIC-04): register completed US-04-02


=== HAZELNUT DATA VALIDATION ===
Train normal: 313
Validation normal: 78
Test official: 110
Batch shape: (2, 3, 256, 256)
Paths: ['hazelnut/train/good/381.png', 'hazelnut/train/good/304.png']
Manifest SHA256: 8f0e0d5b5bb767cd2d2e6f8a0ab10f65a959a97a38bb14bc5d9aae1d30c81fb6

No data leakage: OK
Hazelnut pipeline: OK


In [7]:

import random
import numpy as np
import torch

from anomalib.models.image.patchcore.torch_model import PatchcoreModel

# ============================================
# US-04-04 / EXP-002
# Pooled feature extraction smoke test
# ============================================

SEED = 42
DEVICE = torch.device("cuda:0")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# 1. Initialize pretrained PatchCore backbone
model = PatchcoreModel(
    backbone="wide_resnet50_2",
    layers=["layer2", "layer3"],
    pre_trained=True,
    num_neighbors=1
).to(DEVICE)

# 2. Freeze backbone
model.feature_extractor.requires_grad_(False)
model.eval()

assert not any(
    p.requires_grad
    for p in model.feature_extractor.parameters()
)

# 3. Prepare a normal Hazelnut batch
images = batch["image"].to(DEVICE)

assert images.shape == (2, 3, 256, 256)
assert torch.all(batch["label"] == 0)

torch.cuda.reset_peak_memory_stats()

# 4. Extract features with correct PatchCore pooling
with torch.inference_mode():

    # CNN feature maps
    features = model.feature_extractor(images)

    # Local average pooling (missing in EXP-001)
    pooled_features = {
        layer: model.feature_pooler(tensor)
        for layer, tensor in features.items()
    }

    # Multiscale alignment and concatenation
    embedding_map = model.generate_embedding(
        pooled_features
    )

    # Flatten into patch vectors
    embeddings = model.reshape_embedding(
        embedding_map
    )

    # Copy vectors to CPU
    embeddings_cpu = embeddings.detach().cpu()

# 5. Validate dimensions
assert tuple(features["layer2"].shape) == (
    2, 512, 32, 32
)

assert tuple(features["layer3"].shape) == (
    2, 1024, 16, 16
)

assert tuple(embedding_map.shape) == (
    2, 1536, 32, 32
)

assert tuple(embeddings_cpu.shape) == (
    2048, 1536
)

assert torch.isfinite(embeddings_cpu).all()
assert embeddings_cpu.device.type == "cpu"
assert len(model.embedding_store) == 0

# 6. Report
print("=== PATCHCORE HAZELNUT EXP-002 ===")

print("Input:", tuple(images.shape))

for layer, tensor in features.items():
    print("Raw", layer, tuple(tensor.shape))

for layer, tensor in pooled_features.items():
    print("Pooled", layer, tuple(tensor.shape))

print("Embedding map:", tuple(embedding_map.shape))
print("Flattened embeddings:", tuple(embeddings_cpu.shape))
print("CPU storage:", embeddings_cpu.device)

print(
    "Peak GPU allocated (MiB):",
    round(
        torch.cuda.max_memory_allocated() / (1024 ** 2),
        2
    )
)

print("Embedding store:", len(model.embedding_store))

print("\nPOOLED EXTRACTION SMOKE TEST: PASS")


/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/anomalib/models/image/dinomaly/components/layers.py:22: FutureWarning: The anomalib.models.components.dinov2 package is deprecated and will be removed in a future release. Please use the timm-based feature extractor instead: anomalib.models.components.feature_extractors.TimmFeatureExtractor
  from anomalib.models.components.dinov2.layers import Attention, DropPath, LayerScale, MemEffAttention


model.safetensors: reconstructing file:   0%|          |  0.00B /  276MB            

model.safetensors: downloading bytes:           |  0.00B            

=== PATCHCORE HAZELNUT EXP-002 ===
Input: (2, 3, 256, 256)
Raw layer2 (2, 512, 32, 32)
Raw layer3 (2, 1024, 16, 16)
Pooled layer2 (2, 512, 32, 32)
Pooled layer3 (2, 1024, 16, 16)
Embedding map: (2, 1536, 32, 32)
Flattened embeddings: (2048, 1536)
CPU storage: cpu
Peak GPU allocated (MiB): 134.52
Embedding store: 0

POOLED EXTRACTION SMOKE TEST: PASS


In [8]:

# ============================================
# US-04-04 / EXP-002
# Official PatchCore embedding parity test
# ============================================

# Training mode returns patch embeddings
model.train()

# Keep the pretrained CNN in evaluation mode
model.feature_extractor.eval()

try:
    with torch.inference_mode():
        official_embeddings = model(images)

    assert official_embeddings.shape == embeddings.shape

    max_difference = (
        official_embeddings - embeddings
    ).abs().max().item()

    print("=== PATCHCORE EMBEDDING PARITY ===")
    print("Manual:", tuple(embeddings.shape))
    print("Official:", tuple(official_embeddings.shape))
    print("Maximum difference:", max_difference)

    assert torch.allclose(
        official_embeddings,
        embeddings,
        rtol=1e-5,
        atol=1e-5
    )

    print("\nPATCHCORE EMBEDDING PARITY: PASS")

finally:
    # Remove temporary embeddings stored by forward()
    model.eval()
    model.embedding_store.clear()

assert len(model.embedding_store) == 0


=== PATCHCORE EMBEDDING PARITY ===
Manual: (2048, 1536)
Official: (2048, 1536)
Maximum difference: 0.0

PATCHCORE EMBEDDING PARITY: PASS


In [9]:

import time
import gc
import torch

# ============================================
# US-04-04 / EXP-002 / IDL-50
# Full Hazelnut extraction WITH CORRECT POOLING
# ============================================

SEED = 42
DEVICE = torch.device("cuda:0")
BATCH_SIZE = 2

EXPECTED_IMAGES = 313
PATCHES_PER_IMAGE = 1024
EMBEDDING_DIM = 1536

EXPECTED_VECTORS = EXPECTED_IMAGES * PATCHES_PER_IMAGE

# 1. Release smoke-test tensors
for name in [
    "images",
    "features",
    "pooled_features",
    "embedding_map",
    "embeddings",
    "embeddings_cpu",
    "official_embeddings",
]:
    globals().pop(name, None)

gc.collect()
torch.cuda.empty_cache()

# 2. Verify model state
model.eval()
model.feature_extractor.requires_grad_(False)

assert len(train_dataset) == EXPECTED_IMAGES
assert len(model.embedding_store) == 0, (
    "Unexpected embeddings stored by PatchcoreModel"
)

assert not any(
    p.requires_grad
    for p in model.feature_extractor.parameters()
)

# 3. Recreate DataLoader for deterministic ordering
extraction_loader = create_dataloader(
    train_dataset,
    batch_size=BATCH_SIZE,
    num_workers=0,
    seed=SEED
)

# 4. Preallocate one CPU tensor
embeddings_all_cpu = torch.empty(
    (EXPECTED_VECTORS, EMBEDDING_DIM),
    dtype=torch.float32,
    device="cpu"
)

write_index = 0
processed_images = 0
seen_paths = []

torch.cuda.reset_peak_memory_stats()
start_time = time.perf_counter()

# 5. Extract pooled features batch by batch
for batch_idx, current_batch in enumerate(
    extraction_loader, start=1
):
    paths = list(current_batch["image_path"])
    labels = current_batch["label"]

    assert torch.all(labels == 0)
    assert set(paths).issubset(train_paths)

    with torch.inference_mode():

        current_images = current_batch["image"].to(
            DEVICE
        )

        # STEP A: Raw CNN features
        features = model.feature_extractor(
            current_images
        )

        # STEP B: Correct PatchCore pooling
        # This step was missing in EXP-001
        pooled_features = {
            layer: model.feature_pooler(tensor)
            for layer, tensor in features.items()
        }

        # STEP C: Multiscale alignment + concatenation
        feature_map = model.generate_embedding(
            pooled_features
        )

        b, c, h, w = feature_map.shape

        assert b == len(paths)
        assert c == EMBEDDING_DIM
        assert h * w == PATCHES_PER_IMAGE

        # STEP D: Flatten into patch embeddings
        flat_embeddings = model.reshape_embedding(
            feature_map
        )

        n_vectors = flat_embeddings.shape[0]

        assert tuple(flat_embeddings.shape) == (
            b * PATCHES_PER_IMAGE,
            EMBEDDING_DIM
        )

        assert write_index + n_vectors <= EXPECTED_VECTORS

        # STEP E: GPU -> preallocated CPU tensor
        embeddings_all_cpu[
            write_index:write_index + n_vectors
        ].copy_(flat_embeddings)

    write_index += n_vectors
    processed_images += len(paths)
    seen_paths.extend(paths)

    # Release GPU tensors after each batch
    del current_images
    del features
    del pooled_features
    del feature_map
    del flat_embeddings

    if batch_idx % 20 == 0 or processed_images == EXPECTED_IMAGES:
        print(
            f"Batch {batch_idx}/{len(extraction_loader)} | "
            f"Images {processed_images}/{EXPECTED_IMAGES} | "
            f"Vectors {write_index}/{EXPECTED_VECTORS}"
        )

# 6. Validate full collection
elapsed = time.perf_counter() - start_time

assert processed_images == EXPECTED_IMAGES
assert write_index == EXPECTED_VECTORS

assert len(seen_paths) == EXPECTED_IMAGES
assert len(set(seen_paths)) == EXPECTED_IMAGES
assert set(seen_paths) == train_paths

assert tuple(embeddings_all_cpu.shape) == (
    320512, 1536
)

assert torch.isfinite(
    embeddings_all_cpu
).all()

assert len(model.embedding_store) == 0

# 7. Measure memory usage
gpu_peak_mib = (
    torch.cuda.max_memory_allocated()
    / (1024 ** 2)
)

cpu_tensor_mib = (
    embeddings_all_cpu.numel()
    * embeddings_all_cpu.element_size()
    / (1024 ** 2)
)

# 8. Report
print("\n=== HAZELNUT EXP-002 EXTRACTION REPORT ===")
print("Protocol: PatchCore WITH local pooling")
print("Images processed:", processed_images)
print("Unique paths:", len(set(seen_paths)))
print("Embeddings:", tuple(embeddings_all_cpu.shape))
print("Device:", embeddings_all_cpu.device)
print("Dtype:", embeddings_all_cpu.dtype)
print("CPU tensor size (MiB):", round(cpu_tensor_mib, 2))
print("GPU peak allocated (MiB):", round(gpu_peak_mib, 2))
print("Extraction duration (s):", round(elapsed, 2))
print("Model embedding store:", len(model.embedding_store))

print("\nFull POOLED Hazelnut extraction: PASS")


Batch 20/157 | Images 40/313 | Vectors 40960/320512
Batch 40/157 | Images 80/313 | Vectors 81920/320512
Batch 60/157 | Images 120/313 | Vectors 122880/320512
Batch 80/157 | Images 160/313 | Vectors 163840/320512
Batch 100/157 | Images 200/313 | Vectors 204800/320512
Batch 120/157 | Images 240/313 | Vectors 245760/320512
Batch 140/157 | Images 280/313 | Vectors 286720/320512
Batch 157/157 | Images 313/313 | Vectors 320512/320512

=== HAZELNUT EXP-002 EXTRACTION REPORT ===
Protocol: PatchCore WITH local pooling
Images processed: 313
Unique paths: 313
Embeddings: (320512, 1536)
Device: cpu
Dtype: torch.float32
CPU tensor size (MiB): 1878.0
GPU peak allocated (MiB): 137.02
Extraction duration (s): 38.89
Model embedding store: 0

Full POOLED Hazelnut extraction: PASS


In [10]:

import gc
import random
import numpy as np
import torch

from anomalib.models.components import KCenterGreedy
from sklearn.random_projection import johnson_lindenstrauss_min_dim

# ============================================
# US-04-04 / EXP-002 / IDL-50
# Corrected coreset preflight
# ============================================

SEED = 42
SAMPLING_RATIO = 0.01
EXPERIMENT_ID = "EXP-002"
DEVICE = torch.device("cuda:0")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# 1. Validate pooled embeddings
assert tuple(embeddings_all_cpu.shape) == (320512, 1536)
assert embeddings_all_cpu.device.type == "cpu"
assert embeddings_all_cpu.dtype == torch.float32
assert torch.isfinite(embeddings_all_cpu).all()

assert len(model.embedding_store) == 0
assert not model.training

# Confirm the previous parity test passed
assert "max_difference" in globals()
assert max_difference <= 1e-5

n_vectors, n_features = embeddings_all_cpu.shape

# 2. Initialize sampler without selecting indices
sampler_preview = KCenterGreedy(
    embedding=embeddings_all_cpu,
    sampling_ratio=SAMPLING_RATIO
)

# Deterministic random projection
sampler_preview.model.random_state = SEED

expected_coreset = sampler_preview.coreset_size

projection_dim = int(
    johnson_lindenstrauss_min_dim(
        n_samples=n_vectors,
        eps=sampler_preview.model.eps
    )
)

# 3. GPU memory preflight
free_bytes, total_bytes = torch.cuda.mem_get_info()

def to_mib(value):
    return round(value / (1024 ** 2), 2)

original_bytes = n_vectors * n_features * 4
projected_bytes = n_vectors * projection_dim * 4

print("=== HAZELNUT EXP-002 CORESET PREFLIGHT ===")
print("Experiment:", EXPERIMENT_ID)
print("Protocol: WITH local pooling")
print("Embedding parity difference:", max_difference)
print("Input:", tuple(embeddings_all_cpu.shape))
print("Sampling ratio:", SAMPLING_RATIO)
print("Target coreset:", expected_coreset)
print("Projection epsilon:", sampler_preview.model.eps)
print("Estimated projection dimension:", projection_dim)
print("Random seed:", sampler_preview.model.random_state)

print("\n=== MEMORY ESTIMATION ===")
print("Original embeddings (MiB):", to_mib(original_bytes))
print("Projected features (MiB):", to_mib(projected_bytes))
print("GPU free (MiB):", to_mib(free_bytes))
print("GPU total (MiB):", to_mib(total_bytes))

assert expected_coreset == 3205

# Conservative safety threshold
assert free_bytes >= 6 * (1024 ** 3), (
    "Insufficient GPU memory for coreset selection"
)

del sampler_preview
gc.collect()

print("\nEXP-002 coreset preflight: PASS")


=== HAZELNUT EXP-002 CORESET PREFLIGHT ===
Experiment: EXP-002
Protocol: WITH local pooling
Embedding parity difference: 0.0
Input: (320512, 1536)
Sampling ratio: 0.01
Target coreset: 3205
Projection epsilon: 0.9
Estimated projection dimension: 313
Random seed: 42

=== MEMORY ESTIMATION ===
Original embeddings (MiB): 1878.0
Projected features (MiB): 382.69
GPU free (MiB): 14634.81
GPU total (MiB): 14911.69

EXP-002 coreset preflight: PASS


In [11]:

import gc
import json
import time
import random
import hashlib
from pathlib import Path

import numpy as np
import torch
from anomalib.models.components import KCenterGreedy

# ============================================
# US-04-04 / EXP-002 / IDL-50
# Corrected Hazelnut Memory Bank
# ============================================

SEED = 42
SAMPLING_RATIO = 0.01
EXPERIMENT_ID = "EXP-002"
DEVICE = torch.device("cuda:0")

OUTPUT_DIR = Path(
    "/kaggle/working/patchcore/hazelnut/EXP-002"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Validate pooled extraction
assert tuple(embeddings_all_cpu.shape) == (320512, 1536)
assert embeddings_all_cpu.device.type == "cpu"
assert embeddings_all_cpu.dtype == torch.float32
assert torch.isfinite(embeddings_all_cpu).all()

assert len(model.embedding_store) == 0
assert not model.training
assert max_difference <= 1e-5

assert len(seen_paths) == 313
assert len(set(seen_paths)) == 313
assert set(seen_paths) == train_paths

# 2. Set random seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# 3. Transfer embeddings once to GPU
embedding_gpu = embeddings_all_cpu.to(DEVICE)

sampler = KCenterGreedy(
    embedding=embedding_gpu,
    sampling_ratio=SAMPLING_RATIO
)

sampler.model.random_state = SEED

assert sampler.coreset_size == 3205

print("=== HAZELNUT EXP-002 CORESET SELECTION ===")
print("Protocol: WITH local pooling")
print("Candidates:", embedding_gpu.shape[0])
print("Target size:", sampler.coreset_size)
print("Seed:", SEED)

# 4. Run k-center greedy
torch.cuda.synchronize()
start = time.perf_counter()

selected_idxs = sampler.select_coreset_idxs()

torch.cuda.synchronize()
selection_duration = time.perf_counter() - start

# 5. Validate selected indices
selected_indices_cpu = torch.as_tensor(
    selected_idxs,
    dtype=torch.long,
    device="cpu"
).contiguous()

assert tuple(selected_indices_cpu.shape) == (3205,)
assert torch.unique(selected_indices_cpu).numel() == 3205
assert selected_indices_cpu.min().item() >= 0
assert selected_indices_cpu.max().item() < 320512

# 6. Retrieve original POOLED embeddings
memory_bank_cpu = embeddings_all_cpu.index_select(
    0, selected_indices_cpu
).contiguous()

assert tuple(memory_bank_cpu.shape) == (3205, 1536)
assert torch.isfinite(memory_bank_cpu).all()

# 7. Calculate hashes
def tensor_sha256(tensor):
    array = tensor.detach().cpu().contiguous().numpy()
    return hashlib.sha256(array.tobytes()).hexdigest()

indices_sha256 = tensor_sha256(selected_indices_cpu)
bank_sha256 = tensor_sha256(memory_bank_cpu)

peak_gpu_mib = round(
    torch.cuda.max_memory_allocated() / (1024 ** 2),
    2
)

# 8. Save intermediate coreset artifact
torch.save(
    {
        "memory_bank": memory_bank_cpu,
        "selected_indices": selected_indices_cpu,
        "seed": SEED,
        "sampling_ratio": SAMPLING_RATIO,
        "pooling_corrected": True
    },
    OUTPUT_DIR / "hazelnut_coreset.pt"
)

# 9. Save scientific report
report = {
    "category": "hazelnut",
    "experiment_id": EXPERIMENT_ID,
    "protocol": "PatchCore with local pooling",
    "pooling_corrected": True,
    "embedding_parity_max_difference": float(max_difference),
    "backbone": "wide_resnet50_2",
    "layers": ["layer2", "layer3"],
    "seed": SEED,
    "sampling_ratio": SAMPLING_RATIO,
    "train_images": 313,
    "train_image_paths": seen_paths,
    "input_shape": list(embeddings_all_cpu.shape),
    "memory_bank_shape": list(memory_bank_cpu.shape),
    "extraction_duration_seconds": float(elapsed),
    "selection_duration_seconds": selection_duration,
    "peak_gpu_allocated_mib": peak_gpu_mib,
    "indices_sha256": indices_sha256,
    "memory_bank_sha256": bank_sha256
}

(OUTPUT_DIR / "coreset_report.json").write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

# 10. Release full GPU embeddings
del sampler
del embedding_gpu

gc.collect()
torch.cuda.empty_cache()

# 11. Attach corrected bank
model.memory_bank = memory_bank_cpu.to(DEVICE)
model.eval()

assert tuple(model.memory_bank.shape) == (3205, 1536)
assert len(model.embedding_store) == 0

print("\n=== HAZELNUT EXP-002 MEMORY BANK ===")
print("Selected vectors:", len(selected_idxs))
print("Memory Bank:", tuple(model.memory_bank.shape))
print("Selection duration (s):", round(selection_duration, 2))
print("Peak GPU allocated (MiB):", peak_gpu_mib)
print("Indices SHA256:", indices_sha256)
print("Memory Bank SHA256:", bank_sha256)
print("Saved to:", OUTPUT_DIR)

print("\nCORRECTED HAZELNUT MEMORY BANK: PASS")


=== HAZELNUT EXP-002 CORESET SELECTION ===
Protocol: WITH local pooling
Candidates: 320512
Target size: 3205
Seed: 42


Selecting Coreset Indices.: 100%|██████████| 3204/3204 [00:16<00:00, 198.01it/s]



=== HAZELNUT EXP-002 MEMORY BANK ===
Selected vectors: 3205
Memory Bank: (3205, 1536)
Selection duration (s): 16.65
Peak GPU allocated (MiB): 2756.71
Indices SHA256: 153e19b864d58b94680400ffb466a27c5d5f25fe5363e9174068cbc5f79776d3
Memory Bank SHA256: 2b7cc437fbff34fe137be8b5c19836ae93b92615108d38f7daaa9022e63d9bec
Saved to: /kaggle/working/patchcore/hazelnut/EXP-002

CORRECTED HAZELNUT MEMORY BANK: PASS


In [12]:

import sys
import json
import hashlib
import subprocess
import importlib.metadata as metadata
from pathlib import Path

import torch
import torchvision

# ============================================
# US-04-04 / EXP-002 / IDL-51
# Complete checkpoint save
# ============================================

EXPERIMENT_ID = "EXP-002"

OUTPUT_DIR = Path(
    "/kaggle/working/patchcore/hazelnut/EXP-002"
)

CHECKPOINT_PATH = OUTPUT_DIR / "patchcore_model.pt"

# 1. Verify repository revision
actual_commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True
).strip()

assert actual_commit == SOURCE_COMMIT

# 2. Verify corrected Memory Bank
assert tuple(model.memory_bank.shape) == (3205, 1536)
assert not model.training
assert len(model.embedding_store) == 0
assert max_difference <= 1e-5

coreset_report = json.loads(
    (OUTPUT_DIR / "coreset_report.json").read_text()
)

assert coreset_report["experiment_id"] == EXPERIMENT_ID
assert coreset_report["pooling_corrected"] is True

bank_cpu = model.memory_bank.detach().cpu().contiguous()

bank_hash = hashlib.sha256(
    bank_cpu.numpy().tobytes()
).hexdigest()

assert bank_hash == coreset_report["memory_bank_sha256"]

# 3. Portable CPU state_dict
cpu_state = {
    key: value.detach().cpu().contiguous().clone()
    for key, value in model.state_dict().items()
}

assert "memory_bank" in cpu_state
assert tuple(cpu_state["memory_bank"].shape) == (3205, 1536)

# 4. Experiment metadata
config = {
    "category": "hazelnut",
    "experiment_id": EXPERIMENT_ID,
    "protocol": "PatchCore with local pooling",
    "pooling_corrected": True,
    "embedding_parity_max_difference": float(max_difference),
    "backbone": "wide_resnet50_2",
    "layers": ["layer2", "layer3"],
    "num_neighbors": 1,
    "image_size": [256, 256],
    "sampling_ratio": 0.01,
    "seed": 42,
    "train_images": 313,
    "memory_bank_shape": [3205, 1536],
    "memory_bank_sha256": bank_hash,
    "manifest_sha256": hashlib.sha256(
        MANIFEST.read_bytes()
    ).hexdigest(),
    # Base revision of shared repository code.
    # Notebook EXP-002 must be versioned separately on Kaggle.
    "source_commit": actual_commit,
    "versions": {
        "python": sys.version.split()[0],
        "torch": str(torch.__version__),
        "torchvision": str(torchvision.__version__),
        "anomalib": metadata.version("anomalib"),
        "timm": metadata.version("timm"),
        "lightning": metadata.version("lightning")
    }
}

# 5. Save complete checkpoint
torch.save(
    {
        "state_dict": cpu_state,
        "config": config
    },
    CHECKPOINT_PATH
)

del cpu_state

# 6. File hash helper
def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

checkpoint_hash = file_sha256(CHECKPOINT_PATH)

# 7. Verify safe checkpoint loading
loaded_checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

assert loaded_checkpoint["config"]["experiment_id"] == EXPERIMENT_ID
assert loaded_checkpoint["config"]["pooling_corrected"] is True
assert tuple(
    loaded_checkpoint["state_dict"]["memory_bank"].shape
) == (3205, 1536)

del loaded_checkpoint

print("=== HAZELNUT EXP-002 CHECKPOINT ===")
print("Path:", CHECKPOINT_PATH)
print(
    "Size (MiB):",
    round(CHECKPOINT_PATH.stat().st_size / 1024**2, 2)
)
print("SHA256:", checkpoint_hash)
print("Memory Bank:", tuple(bank_cpu.shape))
print("Git base commit:", actual_commit)
print("Safe weights_only load: PASS")

print("\nEXP-002 checkpoint save: PASS")


=== HAZELNUT EXP-002 CHECKPOINT ===
Path: /kaggle/working/patchcore/hazelnut/EXP-002/patchcore_model.pt
Size (MiB): 113.86
SHA256: 7b4c3784434d69724e8e7fb74d587e4c8a059faa24819dd556c224d03e580a93
Memory Bank: (3205, 1536)
Git base commit: 9c932580cbbe27c61d92905e344c8c7e86457381
Safe weights_only load: PASS

EXP-002 checkpoint save: PASS


In [13]:

import json
import numpy as np
import torch

# ============================================
# US-04-04 / EXP-002 / IDL-51
# GPU reference inference
# ============================================

EXPERIMENT_ID = "EXP-002"

OUTPUT_DIR = Path(
    "/kaggle/working/patchcore/hazelnut/EXP-002"
)

# 1. Build Validation dataset
val_dataset = MVTecDataset(
    dataset_root=DATASET_ROOT,
    manifest_path=MANIFEST,
    category="hazelnut",
    split="val",
    transform=transform
)

assert len(val_dataset) == 78

# 2. Deterministic reference image
sample = val_dataset[0]
reference_path = str(sample["image_path"])

assert reference_path == "hazelnut/train/good/002.png"
assert reference_path in val_paths
assert reference_path not in train_paths
assert int(sample["label"]) == 0

# 3. Confirm corrected model
assert tuple(model.memory_bank.shape) == (3205, 1536)
assert max_difference <= 1e-5

model.eval()

# 4. GPU prediction
with torch.inference_mode():
    input_tensor = sample["image"].unsqueeze(0).to(DEVICE)
    prediction = model(input_tensor)

score = (
    prediction.pred_score.detach()
    .cpu().float().reshape(-1)
)

anomaly_map = (
    prediction.anomaly_map.detach()
    .cpu().float()
)

assert tuple(score.shape) == (1,)
assert tuple(anomaly_map.shape) == (1, 1, 256, 256)
assert torch.isfinite(score).all()
assert torch.isfinite(anomaly_map).all()

# 5. Save reference prediction
reference_file = OUTPUT_DIR / "reference_prediction.npz"

np.savez_compressed(
    reference_file,
    pred_score=score.numpy(),
    anomaly_map=anomaly_map.numpy(),
    image_path=np.array(reference_path)
)

# 6. Verify checkpoint and bank
assert file_sha256(CHECKPOINT_PATH) == checkpoint_hash

coreset_report = json.loads(
    (OUTPUT_DIR / "coreset_report.json").read_text()
)

assert coreset_report["pooling_corrected"] is True

# 7. Save inference report
report = {
    "category": "hazelnut",
    "experiment_id": EXPERIMENT_ID,
    "protocol": "PatchCore with local pooling",
    "pooling_corrected": True,
    "reference_image": reference_path,
    "reference_split": "val",
    "reference_label": 0,
    "pred_score": float(score.item()),
    "anomaly_map_shape": list(anomaly_map.shape),
    "anomaly_map_min": float(anomaly_map.min()),
    "anomaly_map_max": float(anomaly_map.max()),
    "checkpoint_sha256": checkpoint_hash,
    "reference_npz_sha256": file_sha256(reference_file),
    "memory_bank_sha256": bank_hash
}

(OUTPUT_DIR / "inference_report.json").write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("=== HAZELNUT EXP-002 GPU INFERENCE ===")
print("Reference:", reference_path)
print("Split: Validation")
print("Score:", report["pred_score"])
print("Anomaly map:", tuple(anomaly_map.shape))
print(
    "Map range:",
    report["anomaly_map_min"],
    report["anomaly_map_max"]
)
print("Checkpoint SHA256:", checkpoint_hash)
print("Reference saved:", reference_file)

print("\nEXP-002 GPU inference: PASS")


=== HAZELNUT EXP-002 GPU INFERENCE ===
Reference: hazelnut/train/good/002.png
Split: Validation
Score: 47.80107116699219
Anomaly map: (1, 1, 256, 256)
Map range: 17.329195022583008 46.45573806762695
Checkpoint SHA256: 7b4c3784434d69724e8e7fb74d587e4c8a059faa24819dd556c224d03e580a93
Reference saved: /kaggle/working/patchcore/hazelnut/EXP-002/reference_prediction.npz

EXP-002 GPU inference: PASS


In [14]:

import json
import numpy as np
from pathlib import Path

# ============================================
# US-04-04 / EXP-002
# Final GPU artifact validation
# ============================================

EXPERIMENT_ID = "EXP-002"

output_dir = Path(
    "/kaggle/working/patchcore/hazelnut/EXP-002"
)

required_files = [
    "hazelnut_coreset.pt",
    "coreset_report.json",
    "patchcore_model.pt",
    "reference_prediction.npz",
    "inference_report.json",
]

print("=== HAZELNUT EXP-002 ARTIFACTS ===")

# 1. Check all files
for filename in required_files:
    path = output_dir / filename

    assert path.is_file(), f"Missing: {filename}"
    assert path.stat().st_size > 0

    print(
        f"[OK] {filename} "
        f"({path.stat().st_size / 1024**2:.2f} MiB)"
    )

# 2. Verify reports
coreset_data = json.loads(
    (output_dir / "coreset_report.json").read_text()
)

inference_data = json.loads(
    (output_dir / "inference_report.json").read_text()
)

assert coreset_data["experiment_id"] == EXPERIMENT_ID
assert inference_data["experiment_id"] == EXPERIMENT_ID

assert coreset_data["pooling_corrected"] is True
assert inference_data["pooling_corrected"] is True

assert coreset_data["memory_bank_shape"] == [3205, 1536]
assert coreset_data["input_shape"] == [320512, 1536]

# 3. Check file integrity
assert file_sha256(
    output_dir / "patchcore_model.pt"
) == inference_data["checkpoint_sha256"]

assert file_sha256(
    output_dir / "reference_prediction.npz"
) == inference_data["reference_npz_sha256"]

assert (
    coreset_data["memory_bank_sha256"]
    == inference_data["memory_bank_sha256"]
)

# 4. Verify reference prediction
with np.load(
    output_dir / "reference_prediction.npz",
    allow_pickle=False
) as prediction_data:

    reference_score = prediction_data["pred_score"]
    reference_map = prediction_data["anomaly_map"]
    reference_path = str(
        prediction_data["image_path"].item()
    )

    assert reference_score.shape == (1,)
    assert reference_map.shape == (1, 1, 256, 256)

    assert np.isfinite(reference_score).all()
    assert np.isfinite(reference_map).all()

    assert reference_path == (
        "hazelnut/train/good/002.png"
    )

    assert np.isclose(
        float(reference_score[0]),
        inference_data["pred_score"],
        atol=1e-6,
        rtol=0
    )

print("\nExperiment:", EXPERIMENT_ID)
print("Protocol: WITH local pooling")
print("Memory Bank:", coreset_data["memory_bank_shape"])
print("Checkpoint integrity: PASS")
print("Reference prediction integrity: PASS")
print("Pooling correction: VERIFIED")

print("\nALL EXP-002 GPU ARTIFACTS: PASS")


=== HAZELNUT EXP-002 ARTIFACTS ===
[OK] hazelnut_coreset.pt (18.81 MiB)
[OK] coreset_report.json (0.01 MiB)
[OK] patchcore_model.pt (113.86 MiB)
[OK] reference_prediction.npz (0.22 MiB)
[OK] inference_report.json (0.00 MiB)

Experiment: EXP-002
Protocol: WITH local pooling
Memory Bank: [3205, 1536]
Checkpoint integrity: PASS
Reference prediction integrity: PASS
Pooling correction: VERIFIED

ALL EXP-002 GPU ARTIFACTS: PASS
